# Z2005 — Week 03: Linked Lists

A self-study notebook on singly, doubly, and circular linked lists: what they
are, why they exist alongside arrays, and how to build and manipulate them
from scratch in Python.


## Learning Objectives

By the end of this notebook you will be able to:

- Implement a singly linked list from scratch (node class, `push_front`, `push_back`, `insert_after`, delete, traverse, search).
- Implement a doubly linked list and explain why it enables O(1) deletion given a node reference, at the cost of extra memory per node.
- Implement a circular linked list and use it to solve the Josephus problem.
- Compare the time complexity of linked lists against arrays for indexing, insertion, and deletion at different positions.
- Identify and avoid the two classic linked-list bugs: losing the head reference, and off-by-one errors during traversal.
- Solve classic list problems: reversing a list in place and detecting a cycle with Floyd's algorithm.


## How to use this notebook

Run the cells top to bottom. Markdown cells explain a concept; the code cell
right after it is a fully worked, heavily commented example — read the
comments, they explain *why* each line exists, not just what it does.

Cells marked `# TODO` in the **Exercises** section are for you to complete;
replace `raise NotImplementedError` with your own code. Every exercise is
followed by a **Self-Check** cell built from `assert` statements: if your
code is wrong, the cell raises an `AssertionError` (read the message); if
your code is right, it prints a friendly success message and does nothing
else. Do not edit the self-check cells. Solutions are collected at the very
end — try each exercise yourself before you look.


## 1. Why a new data structure at all?

An array (Python's `list`) stores its elements in one contiguous block of
memory, so indexing `arr[i]` is O(1): the computer just adds `i * element_size`
to the start address. That is fast, but it has a cost — inserting or deleting
near the *front* of the array means shifting every element after it, which is
O(n).

A **linked list** takes the opposite trade-off. Instead of one contiguous
block, each element ("node") lives wherever memory happens to have space, and
holds an explicit pointer (`next`) to the following node. There is no
shifting: inserting a new node is just a matter of rewiring a couple of
pointers, in O(1) time, *if you already have a reference to the right spot*.
The price is that you lose O(1) random access — to reach the 500th node you
must walk all 499 nodes before it, which is O(n).

Rule of thumb: arrays win when you read by index a lot; linked lists win when
you insert/delete a lot at positions you already have a reference to (e.g.
the front, or a node you are currently visiting).


In [ ]:
class Node:
    """A single link in the chain: a value plus a pointer to the next node."""
    def __init__(self, value):
        self.value = value
        self.next = None   # None means "this is the last node so far"


# Build a 3-node chain by hand, without any list class yet, to see what
# "linked" really means: each node only knows about the one after it.
n1 = Node(10)
n2 = Node(20)
n3 = Node(30)
n1.next = n2   # wire node 1 -> node 2
n2.next = n3   # wire node 2 -> node 3
# n3.next stays None: n3 is the tail

# Walking the chain manually (this is exactly what traversal does later)
current = n1
values = []
while current is not None:
    values.append(current.value)
    current = current.next   # THE critical step: without this, infinite loop
print(values)  # expect [10, 20, 30]
assert values == [10, 20, 30]


## 2. Singly linked list: a proper class

Now we wrap that node-wiring logic in a `LinkedList` class that keeps track
of the `head` (first node) and a running `_size`, and exposes the operations
you actually need: insert at the front, insert at the back, insert after a
given position, delete by value, traverse to a Python list, and search.

**Common pitfall #1 — losing the head reference.** If you ever reassign
`self.head` without first saving the old head somewhere (or wire a new node
in *before* updating `head`), you can orphan the entire rest of the list —
nothing points to it any more and Python's garbage collector quietly reclaims
it. Always create and fully wire the new node before you touch `head`.

**Common pitfall #2 — off-by-one during traversal.** The loop condition
`while current is not None` (not `while current.next is not None`) is what
lets you visit the *last* node. Mixing these up is the single most common
linked-list bug.


In [ ]:
class LinkedList:
    def __init__(self):
        self.head = None
        self._size = 0

    def __len__(self):
        return self._size

    def is_empty(self):
        return self.head is None

    def push_front(self, value):
        """Insert at the head. O(1): no traversal needed."""
        new_node = Node(value)
        new_node.next = self.head   # wire new node to old head FIRST
        self.head = new_node        # then move head — order matters (pitfall #1)
        self._size += 1

    def push_back(self, value):
        """Insert at the tail. O(n): we must walk to the last node first,
        because a plain singly linked list has no tail pointer."""
        new_node = Node(value)
        if self.head is None:
            self.head = new_node
            self._size += 1
            return
        current = self.head
        while current.next is not None:   # stop AT the last node, not past it
            current = current.next
        current.next = new_node
        self._size += 1

    def insert_after(self, target_value, value):
        """Insert `value` immediately after the first node holding
        `target_value`. O(n) to find the target, O(1) to splice in."""
        current = self.head
        while current is not None and current.value != target_value:
            current = current.next
        if current is None:
            raise ValueError(f"{target_value!r} not found in list")
        new_node = Node(value)
        new_node.next = current.next   # splice AFTER current
        current.next = new_node
        self._size += 1

    def delete(self, value):
        """Delete the first node holding `value`. O(n)."""
        if self.head is None:
            raise ValueError("delete from empty list")
        if self.head.value == value:   # special case: deleting the head
            self.head = self.head.next
            self._size -= 1
            return
        prev, current = self.head, self.head.next
        while current is not None and current.value != value:
            prev, current = current, current.next
        if current is None:
            raise ValueError(f"{value!r} not found in list")
        prev.next = current.next   # unlink current by skipping over it
        self._size -= 1

    def to_list(self):
        """Traverse and collect into a Python list, for easy inspection/testing."""
        result, current = [], self.head
        while current is not None:      # visits every node INCLUDING the last
            result.append(current.value)
            current = current.next
        return result

    def search(self, value):
        """Return True if value is present. O(n) — no shortcut without an index."""
        current = self.head
        while current is not None:
            if current.value == value:
                return True
            current = current.next
        return False


ll = LinkedList()
ll.push_back(10); ll.push_back(20); ll.push_back(30)
ll.push_front(5)
assert ll.to_list() == [5, 10, 20, 30]
ll.insert_after(20, 25)
assert ll.to_list() == [5, 10, 20, 25, 30]
ll.delete(5)   # deletes the head
assert ll.to_list() == [10, 20, 25, 30]
assert ll.search(25) is True
assert ll.search(999) is False
assert len(ll) == 4
print("Singly linked list checks passed")


## 3. Doubly linked list

A doubly linked list adds a second pointer, `prev`, to every node, and a
`tail` pointer to the list itself. The extra pointer costs memory (roughly
one extra reference per node) but buys you two things: you can traverse
backward, and — crucially — you can delete a node in O(1) *given a direct
reference to it*, because you no longer need to walk from the head to find
its predecessor; `node.prev` already tells you.


In [ ]:
class DNode:
    def __init__(self, value):
        self.value = value
        self.next = None
        self.prev = None


class DoublyLinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self._size = 0

    def __len__(self):
        return self._size

    def push_back(self, value):
        """O(1): the tail pointer means no traversal is needed, unlike the
        singly linked list's push_back."""
        new_node = DNode(value)
        if self.tail is None:            # list was empty
            self.head = self.tail = new_node
        else:
            new_node.prev = self.tail    # link backward to old tail
            self.tail.next = new_node    # link old tail forward to new node
            self.tail = new_node         # advance tail
        self._size += 1
        return new_node   # return it so callers can delete it in O(1) later

    def push_front(self, value):
        new_node = DNode(value)
        if self.head is None:
            self.head = self.tail = new_node
        else:
            new_node.next = self.head
            self.head.prev = new_node
            self.head = new_node
        self._size += 1
        return new_node

    def remove_node(self, node):
        """O(1) given a direct node reference: this is the whole point of
        the second pointer. Compare to the singly linked list's delete(),
        which is O(n) because it must search for the predecessor."""
        if node.prev is not None:
            node.prev.next = node.next
        else:
            self.head = node.next        # node was the head
        if node.next is not None:
            node.next.prev = node.prev
        else:
            self.tail = node.prev        # node was the tail
        self._size -= 1

    def to_list_forward(self):
        result, current = [], self.head
        while current is not None:
            result.append(current.value)
            current = current.next
        return result

    def to_list_backward(self):
        result, current = [], self.tail
        while current is not None:
            result.append(current.value)
            current = current.prev
        return result


dll = DoublyLinkedList()
n10 = dll.push_back(10)
n20 = dll.push_back(20)
n30 = dll.push_back(30)
assert dll.to_list_forward() == [10, 20, 30]
assert dll.to_list_backward() == [30, 20, 10]
dll.remove_node(n20)   # O(1): we already hold the node, no search needed
assert dll.to_list_forward() == [10, 30]
assert len(dll) == 2
print("Doubly linked list checks passed")


## 4. Circular linked list and the Josephus problem

In a circular linked list the last node's `next` points back to the first
node instead of to `None` — there is no tail. This models anything arranged
in a ring: a round-robin scheduler cycling through processes, or players
seated in a circle.

The classic circular-list exercise is the **Josephus problem**: `n` people
stand in a circle, numbered `1..n`. Starting from person 1, you count `k`
people around the circle and eliminate the `k`-th; counting continues from
the next survivor. Who is left standing? A circular linked list solves this
directly: skip `k - 1` nodes, then unlink the next one.

**Traversal trap:** because the list has no natural end, `while current is not
None` never terminates on a circular list — it loops forever. You must stop
on a *count* or by comparing back to a known starting node instead.


In [ ]:
def build_circle(n):
    """Build a circular list of nodes valued 1..n and return the head."""
    head = Node(1)
    current = head
    for i in range(2, n + 1):
        current.next = Node(i)
        current = current.next
    current.next = head   # THE circular link: last node points back to head
    return head


def josephus(n, k):
    """Return the value of the last survivor when every k-th person (by
    count) is eliminated from a circle of n people, starting the count at
    person 1."""
    current = build_circle(n)
    # Stop when current points to itself: exactly one node (and hence one
    # self-loop) remains. Using a count/self-loop check, NOT `is not None`,
    # is what avoids the circular-list infinite-loop trap.
    while current.next is not current:
        for _ in range(k - 1):        # walk to the node just BEFORE the one to remove
            current = current.next
        current.next = current.next.next   # unlink the eliminated node
    return current.value


assert josephus(5, 2) == 4
assert josephus(6, 3) == 2
assert josephus(1, 5) == 1   # a circle of one: that person always survives
print("Josephus problem checks passed")


## 5. Arrays vs. linked lists: complexity comparison

| Operation | Array (Python `list`) | Singly linked list | Doubly linked list |
|---|---|---|---|
| Index access `a[i]` | O(1) | O(n) | O(n) |
| Search by value | O(n) | O(n) | O(n) |
| Insert at front | O(n) | O(1) | O(1) |
| Insert at back | O(1) amortized | O(n)* | O(1) |
| Delete at front | O(n) | O(1) | O(1) |
| Delete given a node reference | O(n) | O(n)** | O(1) |
| Extra memory per element | none | 1 pointer | 2 pointers |

\* O(n) unless you keep a separate tail pointer, as the doubly linked list does.
\*\* Even with a node reference, a singly linked list must walk from the head to find the *predecessor*, since a node cannot see backward.

The practical takeaway: reach for a linked list when your workload does a lot
of front-insertion/deletion or splices nodes you already hold a reference to
(e.g. an LRU cache's recency list); reach for an array/list when you mostly
index by position or append at the end.


In [ ]:
import timeit

# Empirically confirm the O(n) vs O(1) front-insertion gap — no invented numbers.
def time_array_front_insert(n):
    arr = []
    def insert_many():
        arr.clear()
        for i in range(n):
            arr.insert(0, i)   # O(n) each time: every existing element shifts right
    return timeit.timeit(insert_many, number=3)

def time_linked_front_insert(n):
    def insert_many():
        ll = LinkedList()
        for i in range(n):
            ll.push_front(i)   # O(1) each time
    return timeit.timeit(insert_many, number=3)

for n in (500, 2000):
    array_time = time_array_front_insert(n)
    linked_time = time_linked_front_insert(n)
    print(f"n={n:5d}  array.insert(0,x): {array_time:.4f}s   linked push_front: {linked_time:.4f}s")

# We don't assert a specific ratio (timing varies by machine), just that
# linked_time doesn't blow up the way array_time does as n grows.


## 6. Reversing a linked list in place

A very common interview/exam question: reverse a singly linked list without
allocating a new list. The trick is to walk forward while re-pointing each
node's `next` to the node *before* it, using three rolling pointers
(`prev`, `current`, `nxt`) so you never lose the rest of the chain.


In [ ]:
def reverse_in_place(linked_list):
    """Reverse a LinkedList in place, O(n) time, O(1) extra space."""
    prev, current = None, linked_list.head
    while current is not None:
        nxt = current.next     # save BEFORE overwriting current.next (pitfall #1's cousin)
        current.next = prev    # reverse this node's pointer
        prev = current          # advance prev
        current = nxt           # advance current using the saved reference
    linked_list.head = prev

reversible = LinkedList()
for v in (1, 2, 3, 4, 5):
    reversible.push_back(v)
reverse_in_place(reversible)
assert reversible.to_list() == [5, 4, 3, 2, 1]
print("In-place reversal check passed")


## 7. Detecting a cycle: Floyd's algorithm

If a bug (or a malicious input) makes a supposedly-linear linked list
circular by accident, naive traversal loops forever. **Floyd's cycle
detection** (the "tortoise and hare") finds a cycle in O(n) time and O(1)
space: run two pointers, one stepping one node at a time (`slow`) and one
stepping two nodes at a time (`fast`). If there is a cycle, `fast` eventually
laps `slow` and they meet; if the list is truly linear, `fast` reaches `None`
first.


In [ ]:
def has_cycle(head):
    """Floyd's tortoise-and-hare cycle detection. O(n) time, O(1) space."""
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next          # one step
        fast = fast.next.next     # two steps
        if slow is fast:          # they can only meet if fast lapped slow, i.e. a cycle exists
            return True
    return False


# A clean, non-circular list: no cycle.
clean = LinkedList()
for v in (1, 2, 3):
    clean.push_back(v)
assert has_cycle(clean.head) is False

# Manually create a cycle: last node points back into the middle.
a, b, c = Node(1), Node(2), Node(3)
a.next, b.next, c.next = b, c, b   # c points back to b: a cycle
assert has_cycle(a) is True

# A fully circular list (Josephus-style) also counts as a cycle.
circular_head = build_circle(4)
assert has_cycle(circular_head) is True

print("Cycle detection checks passed")


## Exercises

Try each one yourself before checking the Solutions section at the end.


**Exercise 1 — `find_middle`.** Given the head of a singly linked list,
return the value of the middle node. If the list has an even number of
nodes, return the value of the *second* of the two middle nodes.
Do it in a single pass using the slow/fast pointer technique from Floyd's
algorithm (no counting the length first).

Example: `[1, 2, 3, 4, 5]` -> `3`. `[1, 2, 3, 4]` -> `3`.


In [ ]:
def find_middle(head):
    """Return the value of the middle node of the list starting at `head`,
    using one pass with slow/fast pointers. For an even-length list, return
    the second of the two middle values.

    Args:
        head: the first Node of a singly linked (non-circular) list. head is
            never None.
    Returns:
        The value stored in the middle node.
    """
    # TODO: implement this using slow/fast pointers, similar to has_cycle above.
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 1
odd = LinkedList()
for v in (1, 2, 3, 4, 5):
    odd.push_back(v)
assert find_middle(odd.head) == 3

even = LinkedList()
for v in (1, 2, 3, 4):
    even.push_back(v)
assert find_middle(even.head) == 3

single = LinkedList()
single.push_back(42)
assert find_middle(single.head) == 42

print("Exercise 1 passed")


**Exercise 2 — `remove_duplicates`.** Given the head of a singly linked
list whose values may repeat, remove duplicate values so that each value
appears only once, keeping the *first* occurrence in place. Do it in place
(rewire `next` pointers; do not build a new list) using a set to track values
already seen.

Example: `[1, 2, 3, 2, 1, 4]` -> `[1, 2, 3, 4]`.


In [ ]:
def remove_duplicates(head):
    """Remove duplicate-valued nodes in place, keeping the first occurrence
    of each value. Returns the (possibly unchanged) head.

    Args:
        head: the first Node of a singly linked list, or None for an empty list.
    Returns:
        The head of the de-duplicated list.
    """
    # TODO: implement this. Hint: keep a `seen` set and a `prev` pointer;
    # when current.value is already in `seen`, splice current out by setting
    # prev.next = current.next instead of advancing prev.
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 2
dup = LinkedList()
for v in (1, 2, 3, 2, 1, 4):
    dup.push_back(v)
new_head = remove_duplicates(dup.head)
result = []
cur = new_head
while cur is not None:
    result.append(cur.value)
    cur = cur.next
assert result == [1, 2, 3, 4]

no_dup = LinkedList()
for v in (1, 2, 3):
    no_dup.push_back(v)
result2 = []
cur = remove_duplicates(no_dup.head)
while cur is not None:
    result2.append(cur.value)
    cur = cur.next
assert result2 == [1, 2, 3]

print("Exercise 2 passed")


**Exercise 3 — `merge_sorted`.** Given the heads of two singly linked
lists that are each already sorted in ascending order, merge them into one
sorted singly linked list and return its head. Do it by rewiring existing
nodes (do not create new `Node` objects). Use a dummy head node to simplify
the edge cases.

Example: `[1, 3, 5]` and `[2, 4, 6]` -> `[1, 2, 3, 4, 5, 6]`.


In [ ]:
def merge_sorted(head_a, head_b):
    """Merge two ascending sorted singly linked lists into one ascending
    sorted list by rewiring nodes, and return the new head.

    Args:
        head_a: head of the first sorted list, or None.
        head_b: head of the second sorted list, or None.
    Returns:
        The head of the merged, sorted list.
    """
    # TODO: implement this. Hint: create `dummy = Node(None)` as a placeholder,
    # keep a `tail` pointer starting at dummy, repeatedly attach the smaller
    # of the two current nodes to tail.next and advance, then attach whatever
    # is left over. Return dummy.next.
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 3
la = LinkedList()
for v in (1, 3, 5):
    la.push_back(v)
lb = LinkedList()
for v in (2, 4, 6):
    lb.push_back(v)
merged_head = merge_sorted(la.head, lb.head)
result = []
cur = merged_head
while cur is not None:
    result.append(cur.value)
    cur = cur.next
assert result == [1, 2, 3, 4, 5, 6]

la2 = LinkedList()
la2.push_back(1)
merged2 = merge_sorted(la2.head, None)
assert merged2.value == 1 and merged2.next is None

print("Exercise 3 passed")


**Exercise 4 (harder) — `josephus_survivors_in_order`.** Extend the
Josephus simulation from Section 4 so that instead of returning only the
final survivor, it returns a list of every eliminated value **in the order
they were eliminated**, followed by the final survivor's value as the last
element.

Example: `n=5, k=2` eliminates 3, 5, 2, 1 in that order, leaving 4 —
so the result is `[3, 5, 2, 1, 4]` (matching `josephus(5, 2) == 4` from
Section 4).


In [ ]:
def josephus_survivors_in_order(n, k):
    """Simulate the Josephus elimination on a circle of n people counting
    by k, and return a list of eliminated values in elimination order, with
    the final survivor's value appended last.

    Args:
        n: number of people in the circle (n >= 1).
        k: count step for elimination (k >= 1).
    Returns:
        A list of length n: eliminated values in order, survivor last.
    """
    # TODO: implement this. Reuse build_circle(n) and the same walking logic
    # as josephus(), but record each eliminated value before unlinking it,
    # and append the survivor's value once only one node remains.
    raise NotImplementedError


In [ ]:
# Self-Check: Exercise 4
assert josephus_survivors_in_order(5, 2) == [3, 5, 2, 1, 4]
assert josephus_survivors_in_order(1, 3) == [1]
result = josephus_survivors_in_order(6, 3)
assert result[-1] == 2   # matches josephus(6, 3) == 2 from Section 4
assert sorted(result) == [1, 2, 3, 4, 5, 6]   # every person appears exactly once
print("Exercise 4 passed")


## Quiz

**Q1.** Why is `push_back` O(n) on a singly linked list but O(1) on a
doubly linked list (as implemented in this notebook)?

<details><summary>Show answer</summary>
The singly linked list has no tail pointer, so push_back must traverse the
entire list to find the last node before it can attach the new one — O(n).
The doubly linked list maintains an explicit `self.tail` reference that is
updated on every insertion, so it can attach directly to the tail in O(1)
without any traversal.
</details>

**Q2.** You are given a direct reference to a node in the *middle* of a
singly linked list (not the head) and asked to delete it in O(1) time
without traversing from the head. Is this possible with only a `next`
pointer, and if so, what's the trick? If not, why not?

<details><summary>Show answer</summary>
It is possible only if the node is not the last node. The trick: instead of
deleting the given node, copy the *next* node's value into it, then delete
the next node by relinking `node.next = node.next.next`. This works because
you never actually needed to touch the predecessor. It fails if the given
node is the tail, since there is no "next" value to copy forward — in that
case you would need the predecessor, which a singly linked list cannot give
you in O(1) without a stored back-pointer.
</details>

**Q3.** In Floyd's cycle detection, why does the fast pointer moving two
steps at a time guarantee it will eventually meet the slow pointer if a
cycle exists, rather than skipping over it forever?

<details><summary>Show answer</summary>
Once both pointers are inside the cycle, the *distance between them* (fast
minus slow, measured along the cycle) increases by exactly 1 node every
step, because fast gains 2 and slow gains 1. Since the cycle has finite
length, this gap increases modulo the cycle length and must eventually hit
0 — meaning fast has "lapped" slow by exactly one full trip around the
cycle, so they land on the same node.
</details>

**Q4.** True or false: for large n, inserting n elements at the *front* of a
Python list (`arr.insert(0, x)`) is asymptotically the same cost as inserting
n elements at the front of a singly linked list with `push_front`.

<details><summary>Show answer</summary>
False. Each `arr.insert(0, x)` call is O(n) because every existing element
must shift right one slot, so n such calls cost O(n^2) total. Each
`push_front` call on a linked list is O(1), so n calls cost O(n) total. This
is exactly what the timing experiment in Section 5 demonstrates empirically.
</details>


## Solutions (try the exercises yourself first!)

Below are reference solutions for all four exercises. Re-running these cells
overwrites the TODO versions above if this notebook is run straight through.


In [ ]:
# Solution: Exercise 1
def find_middle(head):
    slow = fast = head
    while fast is not None and fast.next is not None:
        slow = slow.next
        fast = fast.next.next
    return slow.value

odd = LinkedList()
for v in (1, 2, 3, 4, 5):
    odd.push_back(v)
assert find_middle(odd.head) == 3
even = LinkedList()
for v in (1, 2, 3, 4):
    even.push_back(v)
assert find_middle(even.head) == 3
print("Exercise 1 solution verified")


In [ ]:
# Solution: Exercise 2
def remove_duplicates(head):
    if head is None:
        return None
    seen = {head.value}
    prev, current = head, head.next
    while current is not None:
        if current.value in seen:
            prev.next = current.next   # splice current out
        else:
            seen.add(current.value)
            prev = current              # only advance prev when we keep a node
        current = current.next
    return head

dup = LinkedList()
for v in (1, 2, 3, 2, 1, 4):
    dup.push_back(v)
new_head = remove_duplicates(dup.head)
result = []
cur = new_head
while cur is not None:
    result.append(cur.value)
    cur = cur.next
assert result == [1, 2, 3, 4]
print("Exercise 2 solution verified")


In [ ]:
# Solution: Exercise 3
def merge_sorted(head_a, head_b):
    dummy = Node(None)
    tail = dummy
    a, b = head_a, head_b
    while a is not None and b is not None:
        if a.value <= b.value:
            tail.next = a
            a = a.next
        else:
            tail.next = b
            b = b.next
        tail = tail.next
    tail.next = a if a is not None else b   # attach whatever remains
    return dummy.next

la = LinkedList()
for v in (1, 3, 5):
    la.push_back(v)
lb = LinkedList()
for v in (2, 4, 6):
    lb.push_back(v)
merged_head = merge_sorted(la.head, lb.head)
result = []
cur = merged_head
while cur is not None:
    result.append(cur.value)
    cur = cur.next
assert result == [1, 2, 3, 4, 5, 6]
print("Exercise 3 solution verified")


In [ ]:
# Solution: Exercise 4
def josephus_survivors_in_order(n, k):
    current = build_circle(n)
    eliminated = []
    while current.next is not current:
        for _ in range(k - 1):
            current = current.next
        victim = current.next
        eliminated.append(victim.value)
        current.next = victim.next
    eliminated.append(current.value)   # the final survivor
    return eliminated

assert josephus_survivors_in_order(5, 2) == [3, 5, 2, 1, 4]
assert josephus_survivors_in_order(1, 3) == [1]
print("Exercise 4 solution verified")
